# SUIA/RTH Project LLM — Architecture and Implementation Blueprint

Yes. This is the system we should build. Your description correctly distinguishes the intended Project LLM from a chatbot, a tutorial content generator, or an autonomous AI coding assistant.

Our target is a repository-aware, governance-controlled Block Engineering, Integration, Validation and Certification-Preparation Platform.

The Project LLM will inspect the existing repository, prepare instructions for an external AI, receive its candidate implementation, verify and integrate it into the existing platform, execute validation, assemble evidence, and stop for human certification.

It will not independently approve prototypes, certify blocks, or modify protected infrastructure.

One qualification: this is the target architecture based on the document descriptions discussed so far, not a claim that every current repository implementation detail has been independently verified. Repository discovery and contract reconciliation must precede code changes.

We will proceed in your requested order:

1. Architecture Blueprint — system boundaries, workflows, governance, data, and integration.
2. TypeScript Code Structure — modules, interfaces, database models, APIs, and implementation patterns.
3. End-to-End MVP Plan — execution phases, acceptance criteria, tests, and deliverables.

## Part 1 — Architecture Blueprint

### 1.1 System architecture

Human Architecture Authority

Gate 1 · Prototype approval | Gate 2 · Final certification | Infrastructure decisions

Project LLM Control Plane

Workflow orchestration · Policy enforcement · State management · Human handoffs

Repository Discovery

AST, schemas, Composer, renderer, runtimes

Creation Brief

Template and verified repository facts

Candidate Intake

Files, hashes, manifest, provenance

Integration Engine

Controlled, reviewable repository patches

Validation Engine

Tests, accessibility, SSR, security

Evidence & Audit

Logs, checks, artifacts, certification readiness

External AI

Gemini / Claude / ChatGPT

Manual exchange

Optional Local LLM

Ollama / vLLM

Advisory tasks only initially

Existing SUIA/RTH Platform

Tutorial Composer · Schema · Renderer · UBRC · ILS · LSNB · RSSB

Fundamental architectural rule: Project LLM is an engineering control plane that operates alongside the existing platform. It must not create a parallel Composer, renderer, schema authority, or runtime.

### 1.2 The complete block engineering lifecycle

01

Discover

Inspect repository structure, contracts, existing blocks and runtime integrations.

02

Create brief

Generate a versioned, repository-grounded Creation Brief.

03

External prototype

Human transfers the brief to an external AI; AI produces a prototype.

G1

Human approval

Human approves or rejects the prototype. No automatic bypass.

04

Candidate intake

Accept React/TypeScript files, validate manifest and provenance.

05

Audit and adapt

Check architectural compliance and produce an integration patch.

06

Integrate

Apply permitted changes in an isolated workspace.

07

Validate

Run all applicable checks and record reproducible evidence.

08

Certification readiness

Issue CERTIFICATION_READY only when all mandatory checks pass.

G2

Final certification

Human reviews evidence and makes the final decision.

The workflow must also support `BLOCKED`, `REVISION_REQUIRED`, `REJECTED` and `ESCALATED`. A failing validation must never be silently converted into a pass.

### 1.3 Ownership and responsibility boundaries

| Responsibility                  | Project LLM  | External AI       | Human                  |
| ------------------------------- | ------------ | ----------------- | ---------------------- |
| Repository discovery            | Owns         | —                 | Reviews                |
| Creation Brief                  | Generates    | Receives          | Reviews                |
| Visual prototype                | —            | Creates           | Approves               |
| React candidate                 | Audits       | Creates           | Oversees               |
| Integration patch               | Prepares     | Revises if needed | Authorizes as required |
| Tests and evidence              | Executes     | No authority      | Reviews                |
| Universal infrastructure change | Escalates    | No authority      | Decides                |
| Final certification             | No authority | No authority      | Sole authority         |

### 1.4 Required core engines

I recommend seven modules, implemented incrementally:

| Engine             | Responsibility                              | Initial technology                 |
| ------------------ | ------------------------------------------- | ---------------------------------- |
| Discovery Engine   | Repository inspection and dependency graph  | TypeScript, ts-morph               |
| Brief Engine       | Deterministic brief generation              | TypeScript templates               |
| Intake Engine      | Candidate parsing, hashes, provenance       | Node.js, Zod                       |
| Integration Engine | Safe patch generation and application       | Git worktrees, AST                 |
| Validation Engine  | Test execution and policy checks            | Vitest, Playwright, existing tools |
| Evidence Engine    | Immutable-style audit records and artifacts | PostgreSQL, artifact storage       |
| Workflow Engine    | State transitions and approval gates        | TypeScript, PostgreSQL             |

The implementation should use your existing repository libraries and conventions wherever possible. These are proposed technologies, not verified declarations of what the repository currently uses.

### 1.5 Governance: turn Phase 0 contracts into executable rules

Your existing Phase 0 governance documents should drive policy checks, not simply serve as reference material.

For example:

```
type Decision =
  | { outcome: "ALLOW" }
  | { outcome: "DENY"; reason: string }
  | { outcome: "ESCALATE"; reason: string };

interface PolicyContext {
  actorId: string;
  actorRole: string;
  action: string;
  affectedFiles: string[];
  blockId: string;
  workflowId: string;
}
```

Examples of policy outcomes:

| Proposed operation                      | Outcome  |
| --------------------------------------- | -------- |
| Read an existing block contract         | Allow    |
| Generate a Creation Brief               | Allow    |
| Modify candidate files in sandbox       | Allow    |
| Change shared UBRC infrastructure       | Escalate |
| Change authentication                   | Escalate |
| Approve Gate 1 automatically            | Deny     |
| Issue final certification automatically | Deny     |

Every decision should include the governing rule, its version, the actor, the affected resource, and a timestamp.

### 1.6 Evidence is a first-class artifact

A test result should never be merely:

```
{"status":"PASS"}
```

Instead:

```
{
  "checkId": "block-schema-validation",
  "status": "PASS",
  "candidateHash": "sha256:...",
  "repositoryCommit": "<commit-sha>",
  "validatorVersion": "1.0.0",
  "command": "npm run test:block-schema",
  "exitCode": 0,
  "evidenceArtifactId": "artifact-123",
  "executedAt": "<ISO-timestamp>"
}
```

The system must distinguish between:

- `PASS` — check executed and passed with evidence.
- `FAIL` — check executed and failed.
- `BLOCKED` — prerequisites prevented execution.
- `NOT_APPLICABLE` — documented justification.
- `NOT_RUN` — check has not executed.

`CERTIFICATION_READY` requires all applicable mandatory checks to pass, with any exceptions handled according to an approved policy. It is not equivalent to `CERTIFIED`.

# Part 2 — TypeScript Code Structure

## 2.1 Architecture decision: modular monolith first

Because your earlier documents identify Node.js/TypeScript in the Composer boundary, I recommend a modular TypeScript implementation before introducing independent microservices.

This is a proposed target structure. Do not relocate existing Composer or runtime modules until repository discovery confirms their actual ownership.

```
SUIA_RTH_SHC/
│
├── existing-platform/
│   └── (existing modules remain in place)
│
├── packages/
│   └── project-llm/
│       ├── src/
│       │   ├── domain/
│       │   │   ├── workflow.ts
│       │   │   ├── candidate.ts
│       │   │   ├── evidence.ts
│       │   │   └── approval.ts
│       │   │
│       │   ├── discovery/
│       │   │   ├── repository-scanner.ts
│       │   │   ├── ast-analyzer.ts
│       │   │   ├── dependency-graph.ts
│       │   │   └── architecture-snapshot.ts
│       │   │
│       │   ├── briefs/
│       │   │   ├── brief-builder.ts
│       │   │   └── brief-templates.ts
│       │   │
│       │   ├── intake/
│       │   │   ├── package-parser.ts
│       │   │   ├── manifest-builder.ts
│       │   │   └── provenance.ts
│       │   │
│       │   ├── integration/
│       │   │   ├── patch-planner.ts
│       │   │   ├── patch-validator.ts
│       │   │   └── sandbox-runner.ts
│       │   │
│       │   ├── validation/
│       │   │   ├── schema-validator.ts
│       │   │   ├── runtime-validator.ts
│       │   │   ├── security-validator.ts
│       │   │   └── test-orchestrator.ts
│       │   │
│       │   ├── governance/
│       │   │   ├── policy-engine.ts
│       │   │   └── approval-gates.ts
│       │   │
│       │   ├── evidence/
│       │   │   ├── evidence-ledger.ts
│       │   │   └── certification-report.ts
│       │   │
│       │   ├── models/
│       │   │   ├── model-provider.ts
│       │   │   └── ollama-provider.ts
│       │   │
│       │   └── workflow/
│       │       ├── state-machine.ts
│       │       └── workflow-service.ts
│       │
│       └── tests/
│           ├── unit/
│           ├── integration/
│           ├── security/
│           └── fixtures/
│
└── docs/
    └── project-llm/
```

`existing-platform/` is a conceptual placeholder, not a folder you should create. The real package location must be chosen after inspecting the repository.

## 2.2 Core workflow contract

Use a typed state machine so that no API or agent can skip required approvals.

```
export type WorkflowState =
  | "DISCOVERY"
  | "BRIEF_READY"
  | "AWAITING_PROTOTYPE"
  | "AWAITING_GATE_1"
  | "AWAITING_CANDIDATE"
  | "CANDIDATE_AUDIT"
  | "INTEGRATION"
  | "VALIDATION"
  | "REVISION_REQUIRED"
  | "BLOCKED"
  | "ESCALATED"
  | "CERTIFICATION_READY"
  | "CERTIFIED"
  | "REJECTED";

export interface BlockWorkflow {
  id: string;
  blockFamily: string;
  candidateVersion: number;
  state: WorkflowState;

  repositoryCommit: string;
  contractVersion: string;

  gate1ApprovalId?: string;
  gate2ApprovalId?: string;

  createdAt: string;
  updatedAt: string;
}
```

Important: the transition to `CERTIFIED` must require an independently authenticated human approval record. Changing the `state` property alone must not constitute certification.

## 2.3 Repository discovery contract

The first engine needs to inspect the actual implementation.

```
export interface RepositorySnapshot {
  commitSha: string;
  scannedAt: string;

  composerPaths: string[];
  rendererPaths: string[];
  schemaPaths: string[];
  blockFamilyPaths: string[];

  runtimePaths: {
    ubrc: string[];
    ils: string[];
    lsnb: string[];
    rssb: string[];
  };

  findings: DiscoveryFinding[];
}

export interface DiscoveryFinding {
  id: string;
  category: string;
  description: string;
  sourcePath: string;
  line?: number;
  confidence: "verified" | "inferred";
}
```

Every Creation Brief must reference a repository snapshot. This prevents the brief from describing an architecture that no longer exists.

## 2.4 Creation Brief contract

```
export interface CreationBrief {
  id: string;
  version: number;

  blockFamily: string;
  objective: string;

  repositoryCommit: string;
  architectureSnapshotId: string;

  requiredInterfaces: string[];
  requiredAttributes: string[];

  allowedPaths: string[];
  forbiddenPaths: string[];

  acceptanceCriteria: string[];
  expectedDeliverables: string[];

  status: "DRAFT" | "READY" | "SUPERSEDED";
}
```

A brief should be exportable as a self-contained Markdown document for manual transfer to Gemini, Claude or ChatGPT.

It should include the minimal necessary reference implementation and schemas, rather than the entire repository.

## 2.5 Candidate intake and provenance

```
export interface CandidatePackage {
  id: string;
  briefId: string;
  version: number;

  source: {
    provider: string;
    modelLabel?: string;
    conversationReference?: string;
    submittedBy: string;
  };

  files: CandidateFile[];
  submittedAt: string;
}

export interface CandidateFile {
  relativePath: string;
  sha256: string;
  sizeBytes: number;
  language: string;
}
```

The package manifest must be calculated from the submitted files, not copied from an external AI's claims.

For the first MVP, accept a restricted set of `.tsx`, `.ts`, `.json`, `.css`, and `.md` files. Reject path traversal, symlinks, oversized packages, and unexpected executable or binary content.

## 2.6 Integration engine

Integration should operate through a patch plan, not arbitrary code-writing privileges.

```
export interface IntegrationPlan {
  candidateId: string;
  baseCommit: string;

  operations: Array<{
    path: string;
    operation: "CREATE" | "MODIFY";
    expectedOldHash?: string;
    newHash: string;
  }>;

  policyDecision:
    | "ALLOW"
    | "DENY"
    | "ESCALATE";

  requiresHumanApproval: boolean;
}
```

Recommended sequence:

```
Candidate files
      ↓
Static inspection
      ↓
Architecture compatibility
      ↓
Patch generation
      ↓
Policy evaluation
      ↓
Isolated Git worktree
      ↓
Compile and test
      ↓
Human-reviewable diff
      ↓
Controlled integration
```

Do not run AI-submitted code directly on your developer workstation or production server. Use a disposable isolated environment with no secrets, restricted filesystem access, and network disabled unless a specifically approved test requires it.

## 2.7 Evidence ledger and certification logic

The Evidence Engine should be independent of the optional LLM.

```
export type CheckStatus =
  | "PASS"
  | "FAIL"
  | "BLOCKED"
  | "NOT_RUN"
  | "NOT_APPLICABLE";

export interface ValidationResult {
  checkId: string;
  status: CheckStatus;
  mandatory: boolean;

  evidenceId?: string;
  reason?: string;
}

export function isCertificationReady(
  checks: ValidationResult[]
): boolean {
  return (
    checks.length > 0 &&
    checks.every((check) => {
      if (!check.mandatory) return true;

      return (
        check.status === "PASS" &&
        Boolean(check.evidenceId)
      );
    })
  );
}
```

This is an illustrative predicate only. Production readiness must additionally verify the complete mandatory-check inventory, evidence integrity, approved `NOT_APPLICABLE` decisions, repository commit, candidate hash, Gate 1 approval, and absence of unresolved blockers.

## 2.8 Database design

Use PostgreSQL for transactional workflow records.

| Table                   | Purpose                               |
| ----------------------- | ------------------------------------- |
| `project_llm_workflows` | Workflow identity and current state   |
| `repository_snapshots`  | Commit-pinned architecture discovery  |
| `creation_briefs`       | Versioned external-AI specifications  |
| `candidate_packages`    | Candidate identity and provenance     |
| `candidate_files`       | File manifests and content hashes     |
| `integration_plans`     | Proposed repository changes           |
| `validation_runs`       | Test execution history                |
| `validation_results`    | Individual check outcomes             |
| `evidence_artifacts`    | Logs, screenshots, reports and hashes |
| `human_approvals`       | Gate 1 and Gate 2 decisions           |
| `policy_decisions`      | Governance enforcement history        |
| `workflow_events`       | Append-only state transition audit    |

For the MVP, no vector database is required. Repository discovery is better served initially by AST analysis, structured metadata, deterministic search, and a dependency graph.

Add semantic retrieval only if measured discovery tasks cannot be reliably solved through those methods.

## 2.9 Initial API surface

| Method | Endpoint                                    | Purpose                        |
| ------ | ------------------------------------------- | ------------------------------ |
| POST   | `/api/project-llm/workflows`                | Create workflow                |
| POST   | `/api/project-llm/workflows/:id/discover`   | Inspect repository             |
| POST   | `/api/project-llm/workflows/:id/brief`      | Generate Creation Brief        |
| POST   | `/api/project-llm/workflows/:id/prototype`  | Record prototype submission    |
| POST   | `/api/project-llm/workflows/:id/gate-1`     | Human prototype decision       |
| POST   | `/api/project-llm/workflows/:id/candidates` | Upload candidate package       |
| POST   | `/api/project-llm/workflows/:id/audit`      | Audit candidate                |
| POST   | `/api/project-llm/workflows/:id/integrate`  | Execute controlled integration |
| POST   | `/api/project-llm/workflows/:id/validate`   | Run validation pipeline        |
| GET    | `/api/project-llm/workflows/:id/evidence`   | Retrieve evidence report       |
| POST   | `/api/project-llm/workflows/:id/gate-2`     | Human certification decision   |

All state-changing endpoints need authentication, authorization, idempotency controls where appropriate, and auditable actor identity. Gate endpoints must additionally enforce the required role and valid prior workflow state.

## 2.10 GUI-first workflow

The first interface does not need 12 separate pages. Build one workflow workspace with distinct sections.

Project LLM Workspace

Illustrative interface · SummaryBlock candidate

Draft

DiscoverBriefPrototypeGate 1CandidateAuditIntegrateValidateEvidenceGate 2

Discover

Review repository snapshot, block-family contracts, Composer integration points and runtime dependencies.

Conceptual navigation only; no backend operations are connected.

This gives the user a guided lifecycle instead of exposing raw engineering commands.

# Part 3 — End-to-End MVP Plan

## 3.1 MVP objective

Build one complete, repeatable engineering workflow:

SummaryBlock → Creation Brief → External AI → Prototype Approval → Candidate Intake → Audit → Integration → Validation → Evidence → Human Certification.

For the first version, we are proving the workflow, not attempting to automate every possible block type.

### Proposed six-week execution plan

Week 1

Repository discovery and reconciliation

Map actual Composer, renderer, schemas and runtime dependencies · Resolve lifecycle-versus-generation-service contradiction · Approve canonical architecture and protected-path policy

Approved architecture baseline

Week 2

Deterministic engineering core

Implement repository snapshots · Build Creation Brief templates · Add workflow state machine and audit records

Versioned, reproducible Creation Brief

Week 3

External AI handoff

Create copy/export and candidate upload UI · Implement manifest, provenance and file validation · Enforce human Gate 1

Auditable candidate intake

Week 4

Integration engine

Generate bounded integration patches · Create isolated worktree and sandbox · Implement policy enforcement and escalation

Safe integration candidate

Week 5

Validation and evidence

Connect existing unit/component/integration tests · Add applicable runtime, accessibility, security and SSR checks · Produce evidence ledger and readiness report

Evidence-backed validation

Week 6

Pilot and hardening

Run a real SummaryBlock candidate · Test failures, revisions and unauthorized changes · Perform Gate 2 review and document results

Demonstrated end-to-end workflow

This is a planning estimate, not a guaranteed delivery schedule. Existing test infrastructure, repository complexity, and unresolved architecture decisions may change it.

## 3.2 MVP acceptance criteria

Acceptance checklist

0/10

Repository discovery records the exact commit and verified integration paths

Creation Brief is generated from verified repository facts

Gate 1 cannot be bypassed

Candidate package is independently parsed and hashed

Forbidden modifications are denied or escalated

Integration happens only in an isolated workspace

Required checks produce verifiable evidence

Failed checks generate actionable Revision Requests

CERTIFICATION_READY cannot be issued with missing mandatory evidence

Only an authorized human can certify the block

Use this checklist during implementation and acceptance testing.

## 3.3 What we should deliberately defer

The following are not needed for the first successful pilot: training an LLM, fine-tuning, vector databases, multi-agent orchestration, autonomous repository commits, cloud GPU clusters, or direct paid-model integration.

Once the deterministic workflow is proven, we can benchmark a local Qwen, Llama, or other suitable model for advisory tasks such as explaining audit failures and drafting revision requests. Its output must never override deterministic validation or governance.

## 3.4 The first decision to record

Your repository's reported conflict between the narrower Phase 1B AI generation service and the broader Project LLM lifecycle must be resolved explicitly.

I recommend this decision:

> Architecture Decision PL-001: Project LLM is the engineering and certification-preparation control plane for tutorial block creation and integration. The existing Composer remains the authoritative authoring environment. Any AI-assisted block JSON generation is a separate, optional capability and must not redefine Project LLM's responsibilities.

Mark this as proposed, not approved, until you ratify it and reconcile the existing locked contracts. Preserve the original documents and record supersession relationships rather than silently editing their meaning.

## Final recommendation

Immediate next action: Repository Architecture Audit

Before writing production code, compare the proposed architecture against the actual repository, including its latest commits, Composer, renderer, 18 block families, UBRC/ILS/LSNB/RSSB integrations, and frozen governance contracts.

Start with these three deliverables:

1. `PROJECT_LLM_ARCHITECTURE_RECONCILIATION_REPORT.md`
2. `PROJECT_LLM_CANONICAL_ARCHITECTURE_V1.md`
3. `PROJECT_LLM_MVP_IMPLEMENTATION_CONTRACT.md`

You can connect the GitHub app to work directly with your repository in subsequent steps. Alternatively, a repository ZIP or relevant source files will let me perform a code-grounded audit.

Our first coding milestone should be a working repository discovery engine that produces a verifiable architecture snapshot—not an LLM API call. That establishes the foundation for everything else.